# 7.4 优化步骤①：均衡性（核间负载均衡）

## 1. 木桶效应：整机耗时 = 最慢核的耗时

40 个核并行走完才算结束——39 个核闲着等 1 个长尾核，是最大的浪费。所以优化第一步永远是**均衡性**：让每个 AI Core 分到的负载接近。

负载的度量是 **CoreWeight**：所有 B（batch）× 所有 N（head）的总基本块数 / 总核数，即「每核期望块数 coreWightTarget」。注意这里计的是**有效块**——有 mask 时，被完全遮盖的块（右上角/左下角）不计数。

## 2. 切分维度回顾（第 5 章 → 工程实现）

- B 轴、N 轴：不同 batch / 不同 head 的 QKV 完全独立，随便切；
- S1 轴：Query 按块切后与相同 B/N 的 KV 独立计算，可切；
- S2 轴：softmax 分母是全序列的，默认不切（FlashDecode 才切，见第 5 章 5.4 节）。

## 3. 分核函数：ComputeSplitNBSeq 逐段走读

Tiling 侧的 `RunBigKernelTilingWithParams` 调用 `FADemoSplitNBSeq` 完成分核，核心流程：

**第一步：算有效总面积**。`GetCalcBlockNumsOneHead` 算每个 head 的总块数：无 mask 时 = Q 方向块数 × KV 方向块数（不足一块按一整块算）；有 mask 时 `GetCutBlockNums` 分别算出右上角和左下角被完全遮盖的块数，总面积减去之——**均衡的前提是按真实工作量计数，而不是按形状计数**。

**第二步：算期望负载**。coreWightTarget = 总有效块数 / 核数。

**第三步：贪心切核**。用**六个数组**记录分核结果（每个核的）：S1 块起始/结束、B 起始/结束、N 起始/结束。遍历顺序为三层循环 `for B → for N → for S1块`：

```cpp
for (b : B) for (n : N) for (sOuter : S1Blocks) {
    // 当前行（这个 S1 块）的有效 S2 块数（考虑 mask 边界）
    sInnerBlockNums = GetSInnerBlockNums(...);
    // 离「理想负载 × (当前核+1)」还差多少
    dif = coreWightTarget * (curCore + 1) - curWeight;
    // 当前行块数远超缺口且当前核已有任务 -> 当前核分配完毕，切下一个核
    if (sInnerBlockNums - dif > dif && 已分配) {
        记录 curCore 的起止边界; curCore += 1; 记录新核起点;
    }
    curWeight += sInnerBlockNums;   // 累加负载
}
```

要点：

- **判定条件 `sInnerBlockNums - dif > dif`**：当前行的块数超过剩余缺口的两倍才切核——避免把一行劈到两个核（跨核的行边界处理复杂）；
- **按行分配而不是按块**：每个 S1 块（行）是 softmax 归约的最小完整单元，整行给一个核，核内自己跑 S2 循环；
- **顺序遍历保证连续性**：每核拿到的 (B, N, S1) 区间连续，GM 访问局部性好。

**第四步**：把六个数组写入 TilingData 下发 kernel 侧，kernel 侧按 `GetBlockIdx` 找到自己核的区间直接开算。

## 4. 低并行度场景：FlashDecode

当 KV_length >> Q_length（FIA 的 Q 常为 1，KV 累积到上万 token）且 batch 小时，B×N×S1 的并行度远小于核数——大量核分不到任务。此时把 **S2 也切**：每个核算一段 KV 的局部 m/l/输出，再由 combine 归并（第 5 章 5.4 节的数学，第 9 章的工程细节）。

## 小测验

1. CoreWeight 是怎么定义的？为什么 mask 场景必须按「有效块」而不是「形状块」计数？
2. 六个数组分别记录什么？为什么按「行」（S1 块）分配而不是把块均分？
3. 切核判定 `sInnerBlockNums - dif > dif` 的含义是什么？它避免了什么问题？
4. 什么场景下必须切 S2（FlashDecode）？付出的代价是什么？

In [ ]:
# 动手：模拟 ComputeSplitNBSeq 的贪心分核（含 mask 有效块）
import numpy as np

n_cores = 8
S1_blocks, S2_blocks_full = 10, 12
# 模拟因果 mask：第 i 个 S1 块的有效 S2 块数（对角线以下）
rows = [ (i * S2_blocks_full) // S1_blocks + 1 for i in range(S1_blocks) ]
total = sum(rows)
target = total / n_cores
print('每行有效块数:', rows, ' 总数:', total, ' 每核期望:', round(target, 2))

cores, cur, weight = [], [], 0
cur_start = 0
for i, r in enumerate(rows):
    dif = target * (len(cores) + 1) - weight
    if r - dif > dif and cur:               # 超过缺口两倍且当前核非空 -> 切核
        cores.append((cur_start, i)); cur_start = i
        cur = []
    cur.append(i); weight += r
cores.append((cur_start, S1_blocks))

loads = [sum(rows[a:b]) for a, b in cores]
print('分核区间:', cores)
print('各核负载:', loads)
print(f'最差核/期望 = {max(loads)/target:.2f}（越接近 1 越均衡）')
print()
print('观察：按行整块分配（不劈行）的代价是负载有离散抖动；')
print('     行粒度越细（S1 块越多），抖动越小——这就是 BN1S1 细粒度分核优于 BN1 的原因。')